In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os

from tqdm import tqdm
import torch
import torch.nn.functional as F
import numpy as np

from encoder import ConvEncoder
from contextualizer import Contextualizer
from Model.BENDRTrainer import BENDRTrainer
from dataset import EEGDataset
from transforms import RandomTemporalCrop

In [7]:
pretrain_args = {
  "mask_rate": 0.065,
  "mask_span": 10,
  "layer_drop": 0.01,
  "multi_gpu": True,
  "temp": 0.1,
  "encoder_grad_frac": 0.1,
  "num_negatives": 20,
  "enc_feat_l2": 1.0
}

In [8]:
encoder = ConvEncoder(19, encoder_h=512)
contextualizer = Contextualizer(encoder.encoder_h, layer_drop=0.01)
trainer = BENDRTrainer(encoder, contextualizer, **pretrain_args)

In [9]:
# Slower learning rate for the encoder
trainer.set_optimizer(torch.optim.Adam(trainer.parameters()))
trainer.add_batch_transform(RandomTemporalCrop(max_crop_frac=0.05))

In [10]:
def load_dataset(path):
    ecs = []
    eos = []

    for subject in tqdm(os.listdir(path)):
        ec = torch.load(os.path.join(path, subject, 'EC.pt'), weights_only=True)
        eo = torch.load(os.path.join(path, subject, 'EO.pt'), weights_only=True)

        ecs.append(ec)
        eos.append(eo)

    ecs = torch.cat(ecs, axis=0).float()
    eos = torch.cat(eos, axis=0).float()

    dataset = torch.cat([ecs, eos], axis=0)
    print("Dataset shape:", dataset.shape)

    return dataset

In [11]:
dataset = load_dataset('/home/azureuser/mycontainer/SmallSubset')
print(dataset[0][0].shape, dataset[0][0].dtype)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:02<00:00,  7.13it/s]


Dataset shape: torch.Size([80, 19, 15360])
torch.Size([15360]) torch.float32


In [20]:
trainer.fit(training_dataset=dataset, epochs=5, batch_size=8)

Loading data with 4 additional workers


Iteration:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/10 [00:00<?, ?batches/s]/

RuntimeError: Expected weight to be a vector of size equal to the number of channels in input, but got weight of shape [512] and input of shape [512, 5120]